# M5 — CV/Transcript NER Extractor (BERT-base-uncased)
Target: **Entity F1 >= 0.82** (on the gold test split). See `docs/phases/MENTORA_Phase4_Model_Training.md` §5.

**Known limitation carried over from Phase 3** (`mentora_data/SOURCES.md`):
the gold set (`gold_val_test.jsonl`) is auto-tagged, not yet hand-corrected.
Training on silver / evaluating on gold is still the right discipline (it's
the best split available), but until someone manually reviews
`mentora_data/labeled/m5/gold_review.csv`, the reported F1 reflects
auto-tagger quality on both sides, not auto-tagger-vs-human-truth. Schema
(`tokens`, `ner_tags` per line) matches the plan doc exactly — no adaptation
needed here.

## Setup — mount Drive, confirm GPU, install deps
Same first 3 cells in every Mentora training notebook. See `docs/phases/MENTORA_Phase4_Model_Training.md` §0.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

In [ ]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate evaluate seqeval

## Load gold (val/test) and silver (train) sets

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

silver = load_jsonl(f'{DATA}/labeled/m5/silver_train.jsonl')
gold = load_jsonl(f'{DATA}/labeled/m5/gold_val_test.jsonl')
print(f"{len(silver)} silver (train), {len(gold)} gold (val+test)")

all_tags = sorted({tag for row in silver + gold for tag in row['ner_tags']})
tag2id = {t: i for i, t in enumerate(all_tags)}
id2tag = {i: t for t, i in tag2id.items()}
print("tags:", all_tags)

import random
random.seed(42)
random.shuffle(gold)
val, test = gold[:len(gold)//2], gold[len(gold)//2:]

train_ds = Dataset.from_list(silver)
val_ds = Dataset.from_list(val)
test_ds = Dataset.from_list(test)

## Tokenize with label alignment (sub-token labels for BERT's wordpiece tokenizer)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_and_align(batch):
    tokenized = tokenizer(batch['tokens'], truncation=True, is_split_into_words=True, padding='max_length', max_length=256)
    all_labels = []
    for i, tags in enumerate(batch['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids, prev_word = [], None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(tag2id[tags[word_id]])
            else:
                label_ids.append(-100)
            prev_word = word_id
        all_labels.append(label_ids)
    tokenized['labels'] = all_labels
    return tokenized

train_ds = train_ds.map(tokenize_and_align, batched=True)
val_ds = val_ds.map(tokenize_and_align, batched=True)
test_ds = test_ds.map(tokenize_and_align, batched=True)

## Model, metrics (entity-level F1 via seqeval), training

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-uncased', num_labels=len(all_tags), id2label=id2tag, label2id=tag2id
)
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)
    true_preds, true_labels = [], []
    for pred_row, label_row in zip(preds, labels):
        p_seq, l_seq = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                p_seq.append(id2tag[p]); l_seq.append(id2tag[l])
        true_preds.append(p_seq); true_labels.append(l_seq)
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {"entity_f1": results["overall_f1"]}

OUT_DIR = f'{MODELS}/model5_cv_ner'
args = TrainingArguments(
    output_dir=OUT_DIR, per_device_train_batch_size=8, num_train_epochs=4,
    save_strategy="epoch", eval_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="entity_f1", fp16=True,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)

import glob
existing = sorted(glob.glob(f'{OUT_DIR}/checkpoint-*'))
trainer.train(resume_from_checkpoint=existing[-1] if existing else None)

print("Test set (gold, held out):", trainer.evaluate(test_ds))
trainer.save_model(f'{OUT_DIR}/final')
tokenizer.save_pretrained(f'{OUT_DIR}/final')

## If you miss the target (Entity F1 < 0.82)
- Check per-entity-type F1 — CERT usually drags the average down (smallest
  auto-tagger vocabulary in Phase 3)
- Silver-label noise is the other usual culprit — tighten the PhraseMatcher
  vocabulary and regenerate the silver set if spot-checks show high error
- **Most importantly: do the manual gold-set review first** (see the note at
  the top) — training harder against noisy gold labels has a low ceiling